# Hunyuan Machine Translation with OpenVINO

[HY-MT1.5](https://huggingface.co/tencent/HY-MT1.5-1.8B) is a series of machine translation models developed by Tencent. Built on the Hunyuan Dense V1 architecture, these models deliver high-quality translation across **33+ languages** including Chinese, English, Japanese, Korean, French, German, Spanish, Arabic, and more.

Two model sizes are available:
- **[HY-MT1.5-1.8B](https://huggingface.co/tencent/HY-MT1.5-1.8B)** — Lightweight, suitable for resource-constrained environments
- **[HY-MT1.5-7B](https://huggingface.co/tencent/HY-MT1.5-7B)** — Larger model with higher translation quality

In this tutorial, we will:

- Install prerequisites
- Select and convert the model to OpenVINO IR format using [Optimum Intel](https://huggingface.co/docs/optimum/intel/index)
- Apply weight compression (INT4/INT8/FP16) using [NNCF](https://github.com/openvinotoolkit/nncf)
- Run translation inference with [OpenVINO Generate API](https://github.com/openvinotoolkit/openvino.genai)
- Launch an interactive Gradio translation demo

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Select model](#Select-model)
- [Convert and compress model](#Convert-and-compress-model)
- [Select device for inference](#Select-device-for-inference)
- [Run translation pipeline](#Run-translation-pipeline)
- [Interactive translation demo](#Interactive-translation-demo)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO. It may be fully supported and validated in the future.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/hunyuan-translation/hunyuan-translation.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)

Install required dependencies

In [1]:
import os
import platform

os.environ["GIT_CLONE_PROTECTION_ACTIVE"] = "false"

%pip install -q -U openvino-genai
%pip install -q --extra-index-url https://download.pytorch.org/whl/cpu \
"git+https://github.com/huggingface/optimum-intel.git" \
"nncf>=2.18.0" \
"torch>=2.8" \
"accelerate" \
"gradio>=6.0" \
"transformers>=4.57.0" \
"huggingface-hub"

if platform.system() == "Darwin":
    %pip install -q "numpy<2.0.0"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import requests

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w", encoding="utf-8").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("hunyuan-translation.ipynb")

## Select model
[back to top ⬆️](#Table-of-contents:)

Select the Hunyuan translation model and weight compression option. The 1.8B model is recommended for most use cases due to lower resource requirements. The 7B model provides higher translation quality but requires more memory.

In [3]:
import ipywidgets as widgets

model_options = {
    "HY-MT1.5-1.8B": "tencent/HY-MT1.5-1.8B",
    "HY-MT1.5-7B": "tencent/HY-MT1.5-7B",
}

compression_options = ["INT4", "INT8", "FP16"]

model_selector = widgets.Dropdown(
    options=model_options,
    value="tencent/HY-MT1.5-1.8B",
    description="Model:",
)

compression_selector = widgets.Dropdown(
    options=compression_options,
    value="INT4",
    description="Compression:",
)

display(model_selector, compression_selector)

Dropdown(description='Model:', options={'HY-MT1.5-1.8B': 'tencent/HY-MT1.5-1.8B', 'HY-MT1.5-7B': 'tencent/HY-M…

Dropdown(description='Compression:', options=('INT4', 'INT8', 'FP16'), value='INT4')

In [4]:
model_id = model_selector.value
model_name = model_selector.label
compression = compression_selector.value

print(f"Selected model: {model_name} ({model_id})")
print(f"Compression: {compression}")

Selected model: HY-MT1.5-1.8B (tencent/HY-MT1.5-1.8B)
Compression: INT4


## Convert and compress model
[back to top ⬆️](#Table-of-contents:)

Convert the model to OpenVINO IR format using [Optimum Intel](https://huggingface.co/docs/optimum/intel/index). The `optimum-cli export openvino` command handles the conversion and optional weight compression in a single step.

<details>
  <summary><b>Click here to read more about Optimum CLI usage</b></summary>

The basic command for model export:

```bash
optimum-cli export openvino --model <model_id> --task text-generation-with-past --weight-format <format> <output_dir>
```

Where:
- `--model` is the model id from HuggingFace Hub or local path
- `--task text-generation-with-past` enables KV-cache for efficient inference
- `--weight-format` can be `fp16`, `int8`, or `int4` for compression

For INT4, you can further tune with `--group-size` and `--ratio` parameters.
</details>

In [5]:
from pathlib import Path
from cmd_helper import optimum_cli

model_dir = Path(model_name) / compression

if not model_dir.exists():
    weight_format = compression.lower()

    additional_args = {
        "task": "text-generation-with-past",
        "weight-format": weight_format,
    }

    if weight_format == "int4":
        additional_args["ratio"] = "1.0"
        additional_args["group-size"] = "128"

    optimum_cli(model_id, model_dir, additional_args=additional_args)
else:
    print(f"Model already exists at {model_dir}, skipping conversion.")

Model already exists at HY-MT1.5-1.8B\INT4, skipping conversion.


In [6]:
from pathlib import Path


def get_model_size(model_path):
    """Calculate total size of model files in MB."""
    total = 0
    for f in Path(model_path).glob("**/*"):
        if f.is_file() and f.suffix in (".bin", ".xml"):
            total += f.stat().st_size
    return total / (1024 * 1024)


if model_dir.exists():
    size_mb = get_model_size(model_dir)
    print(f"Model size ({compression}): {size_mb:.1f} MB")

Model size (INT4): 1011.1 MB


## Select device for inference
[back to top ⬆️](#Table-of-contents:)

Select the device to run inference on.

In [7]:
from notebook_utils import device_widget

device = device_widget(default="CPU")

device

Dropdown(description='Device:', options=('CPU', 'GPU', 'NPU', 'AUTO'), value='CPU')

## Run translation pipeline
[back to top ⬆️](#Table-of-contents:)

We use [OpenVINO Generate API](https://github.com/openvinotoolkit/openvino.genai) to create the inference pipeline. The `LLMPipeline` loads the converted model and provides a simple interface for text generation.

HY-MT1.5 uses specific prompt templates for translation:
- **Chinese ↔ Any language**: Chinese prompt format
- **Non-Chinese ↔ Non-Chinese**: English prompt format

Let's first test a quick translation to verify the pipeline works.

In [8]:
import openvino_genai as ov_genai
import sys

print(f"Loading model from {model_dir}...\n")

pipe = ov_genai.LLMPipeline(str(model_dir), device.value)

Loading model from HY-MT1.5-1.8B\INT4...



In [9]:
# Supported language names for prompt formatting
LANGUAGE_MAP = {
    "Chinese": "中文",
    "English": "English",
    "French": "French",
    "German": "German",
    "Spanish": "Spanish",
    "Japanese": "Japanese",
    "Korean": "Korean",
    "Russian": "Russian",
    "Arabic": "Arabic",
    "Portuguese": "Portuguese",
    "Italian": "Italian",
    "Thai": "Thai",
    "Vietnamese": "Vietnamese",
    "Turkish": "Turkish",
    "Indonesian": "Indonesian",
    "Malay": "Malay",
    "Hindi": "Hindi",
    "Polish": "Polish",
    "Dutch": "Dutch",
    "Czech": "Czech",
    "Filipino": "Filipino",
    "Bengali": "Bengali",
    "Tamil": "Tamil",
    "Ukrainian": "Ukrainian",
    "Persian": "Persian",
    "Hebrew": "Hebrew",
    "Urdu": "Urdu",
    "Khmer": "Khmer",
    "Burmese": "Burmese",
    "Gujarati": "Gujarati",
    "Telugu": "Telugu",
    "Marathi": "Marathi",
    "Traditional Chinese": "繁体中文",
}

# Chinese language identifiers for prompt selection
CHINESE_LANGUAGES = {"Chinese", "Traditional Chinese"}


def build_translation_prompt(source_text, source_lang, target_lang):
    """Build the translation prompt following HY-MT1.5 format."""
    target_name = LANGUAGE_MAP.get(target_lang, target_lang)

    # Use Chinese prompt if either language is Chinese
    if source_lang in CHINESE_LANGUAGES or target_lang in CHINESE_LANGUAGES:
        prompt = f"将以下文本翻译为{target_name}，注意只需要输出翻译后的结果，不要额外解释：\n\n{source_text}"
    else:
        prompt = f"Translate the following segment into {target_name}, without additional explanation.\n\n{source_text}"

    return prompt


# Test translation: English -> Chinese
test_text = "OpenVINO is a toolkit for optimizing and deploying deep learning models."
prompt = build_translation_prompt(test_text, "English", "Chinese")

print(f"Source (English): {test_text}")
print("\nTranslation (Chinese):")

generation_config = ov_genai.GenerationConfig()
generation_config.max_new_tokens = 256


def streamer(subword):
    print(subword, end="", flush=True)
    sys.stdout.flush()
    return False


result = pipe.generate(prompt, generation_config, streamer)

Source (English): OpenVINO is a toolkit for optimizing and deploying deep learning models.

Translation (Chinese):
OpenVINO是一种用于优化和部署深度学习模型的工具包。

In [10]:
# Test translation: Chinese -> English
test_text_cn = "人工智能正在深刻改变我们的生活方式和工作方式。"
prompt_cn = build_translation_prompt(test_text_cn, "Chinese", "English")

print(f"Source (Chinese): {test_text_cn}")
print("\nTranslation (English):")

result = pipe.generate(prompt_cn, generation_config, streamer)

Source (Chinese): 人工智能正在深刻改变我们的生活方式和工作方式。

Translation (English):
Artificial intelligence is profoundly changing the way we live and work.

## Interactive translation demo
[back to top ⬆️](#Table-of-contents:)

Launch an interactive Gradio demo for real-time translation. The demo allows you to:
- Select source and target languages from 33+ supported options
- Enter text and get instant translations
- Adjust generation parameters (temperature, top-k, top-p, repetition penalty)
- Try pre-built translation examples

In [ ]:
from gradio_helper import make_demo

demo = make_demo(pipe, model_name)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)
# If you are launching remotely, specify server_name and server_port
# EXAMPLE: `demo.launch(server_name='your server name', server_port='server port in int')`
# To learn more please refer to the Gradio docs: https://gradio.app/docs/